In [ ]:
!pip install vaderSentiment

In [ ]:
# STEP 1: LOAD CLEANED DATA
# We load the preprocessed CSV produced by the preprocessing notebook.
# The clean_text column is used as input for both sentiment models.
import pandas as pd
df = pd.read_csv('youtube_comments_preprocessedfinal.csv')

df.head()

In [ ]:
# STEP 2: VADER SENTIMENT ANALYSIS
# VADER (Valence Aware Dictionary and sEntiment Reasoner) is a lexicon-based
# model designed for social media text. It produces a compound score from -1 to 1.
# Scores >= 0.05 are labelled positive, <= -0.05 negative, and the rest neutral.
# VADER is fast and requires no GPU, making it a practical baseline.
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm import tqdm

analyzer = SentimentIntensityAnalyzer()

tqdm.pandas()

def get_sentiment(text):
    scores = analyzer.polarity_scores(text)

    compound = scores["compound"]

    if compound >= 0.05:
        return "Positive"
    elif compound <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df["sentiment"] = df["clean_text"].progress_apply(get_sentiment)

In [ ]:
def get_compound_score(text):
    return analyzer.polarity_scores(text)["compound"]

df["sentiment_score"] = df["clean_text"].progress_apply(get_compound_score)

In [ ]:
print(df["sentiment"].value_counts())

In [ ]:
sentiment_percentages = (
    df["sentiment"]
    .value_counts(normalize=True)
    * 100
)

print(sentiment_percentages)

In [ ]:
import matplotlib.pyplot as plt

df["sentiment"].value_counts().plot(
    kind="bar"
)

plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Comments")

plt.show()

In [ ]:
df[
    ["clean_text", "sentiment"]
].sample(20)

In [ ]:
df.rename(columns={
    "sentiment":"vader_sentiment",
    "sentiment_score":"vader_score"
}, inplace=True)

In [ ]:
!pip install transformers torch tqdm

In [ ]:
from transformers import pipeline
from tqdm import tqdm
import pandas as pd

tqdm.pandas()

sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length=512
)

In [ ]:
# STEP 3: ROBERTA SENTIMENT ANALYSIS
# We use cardiffnlp/twitter-roberta-base-sentiment-latest from HuggingFace.
# This model was fine-tuned on Twitter data, which shares informal language
# characteristics with YouTube comments (abbreviations, slang, short sentences).
# It is run via the HuggingFace pipeline with truncation at 512 tokens.
# This step takes several minutes on the full dataset — GPU is recommended.
sample_df = df.sample(1000, random_state=42).copy()

def get_roberta_sentiment(text):
    result = sentiment_model(str(text))[0]
    return result["label"]

sample_df["roberta_sentiment"] = sample_df["clean_text"].progress_apply(get_roberta_sentiment)

sample_df[["clean_text", "roberta_sentiment"]].sample(10)

In [ ]:
def get_roberta_sentiment(text):
    result = sentiment_model(str(text))[0]
    return result["label"]

df["roberta_sentiment"] = df["clean_text"].progress_apply(get_roberta_sentiment)

In [ ]:
def get_roberta_score(text):
    result = sentiment_model(str(text))[0]
    return result["score"]

df["roberta_score"] = df["clean_text"].progress_apply(get_roberta_score)

In [ ]:
print(df["roberta_sentiment"].value_counts())

In [ ]:
print(
    (df["roberta_sentiment"].value_counts(normalize=True) * 100)
    .round(2)
)

In [ ]:
import matplotlib.pyplot as plt

df["roberta_sentiment"].value_counts().plot(kind="bar")

plt.title("RoBERTa Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Comments")

plt.show()

In [ ]:
df[df["roberta_sentiment"]=="positive"][
    ["clean_text","roberta_score"]
].sample(10)

In [ ]:
df[df["roberta_sentiment"]=="negative"][
    ["clean_text","roberta_score"]
].sample(10)

In [ ]:
df[df["roberta_sentiment"]=="neutral"][
    ["clean_text","roberta_score"]
].sample(10)

In [ ]:

df[
    ["clean_text", "roberta_sentiment"]
].sample(20)

In [ ]:
df.to_csv("youtube_comments_sentiment.csv", index=False)

In [ ]:
print(df.shape)
print(df.columns)

In [ ]:
from google.colab import files

files.download("youtube_comments_sentiment.csv")

In [ ]:
# STEP 5: MANUAL EVALUATION SETUP
# We sample 100 comments and export them for manual labelling.
# Each comment was independently labelled by the team as positive, negative, or neutral.
# The labelled file was re-imported and used to calculate accuracy for both models.
evaluation_df = df.sample(100, random_state=42).copy()

evaluation_df[[
    "clean_text",
    "vader_sentiment",
    "roberta_sentiment"
]].to_csv("manual_sentiment_evaluation.csv", index=False)

In [ ]:
from google.colab import files
files.download("manual_sentiment_evaluation.csv")

In [ ]:
import pandas as pd

eval_df = pd.read_csv("manual_sentiment_evaluation_labelled.csv")

In [ ]:
eval_df["human_sentiment"] = (
    eval_df["human_sentiment"]
    .str.lower()
    .str.strip()
)

eval_df["vader_sentiment"] = (
    eval_df["vader_sentiment"]
    .str.lower()
    .str.strip()
)

eval_df["roberta_sentiment"] = (
    eval_df["roberta_sentiment"]
    .str.lower()
    .str.strip()
)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

vader_acc = accuracy_score(eval_df["human_sentiment"], eval_df["vader_sentiment"])
roberta_acc = accuracy_score(eval_df["human_sentiment"], eval_df["roberta_sentiment"])

print("VADER Accuracy:", round(vader_acc, 3))
print("RoBERTa Accuracy:", round(roberta_acc, 3))

In [ ]:
print("VADER Report")
print(classification_report(eval_df["human_sentiment"], eval_df["vader_sentiment"]))

print("RoBERTa Report")
print(classification_report(eval_df["human_sentiment"], eval_df["roberta_sentiment"]))

In [ ]:
# STEP 7: CONFUSION MATRICES
# The heatmaps visualise which sentiment classes each model confuses most.
# VADER's main failure: classifying neutral comments as positive due to
# isolated positive words even when the overall comment is neutral or mixed.
# RoBERTa's main failure: sarcasm and very short comments with no clear context.
import seaborn as sns
import matplotlib.pyplot as plt

labels = ["negative", "neutral", "positive"]

cm_roberta = confusion_matrix(
    eval_df["human_sentiment"],
    eval_df["roberta_sentiment"],
    labels=labels
)

sns.heatmap(
    cm_roberta,
    annot=True,
    fmt="d",
    xticklabels=labels,
    yticklabels=labels
)

plt.title("RoBERTa Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Human Label")
plt.show()

In [ ]:
cm_vader = confusion_matrix(
    eval_df["human_sentiment"],
    eval_df["vader_sentiment"],
    labels=labels
)

sns.heatmap(
    cm_vader,
    annot=True,
    fmt="d",
    xticklabels=labels,
    yticklabels=labels
)

plt.title("VADER Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Human Label")
plt.show()